In [9]:
'''
orchestrating teams of specialized agents to tackle complex, multi-step problems that a single agent cannot handle alone.

ROUTER AGENT:
"master" router agent that intelligently analyzes a user's request and delegates it to the correct agent or workflow.

SEQUENTIAL Agent:

creating pipelines where the output of one agent becomes the input for the next.

Some tasks are too complex for one agent. A user might ask, "Find me a great restaurant and then tell me how to get there." 
This requires two different skills: food recommendation and navigation.

We'll build a system that can handle this by:
1.  Creating a new `transportation_agent`.
2.  Teaching our `router_agent` to recognize these special "combo" requests.
3.  Writing Python code (the "orchestrator") that runs the agents in a sequence, passing the output of the first agent to the second.
'''


'\norchestrating teams of specialized agents to tackle complex, multi-step problems that a single agent cannot handle alone.\n\nSEQUENTIAL Agent:\n\ncreating pipelines where the output of one agent becomes the input for the next.\n\nSome tasks are too complex for one agent. A user might ask, "Find me a great restaurant and then tell me how to get there." \nThis requires two different skills: food recommendation and navigation.\n\nWe\'ll build a system that can handle this by:\n1.  Creating a new `transportation_agent`.\n2.  Teaching our `router_agent` to recognize these special "combo" requests.\n3.  Writing Python code (the "orchestrator") that runs the agents in a sequence, passing the output of the first agent to the second.\n'

In [10]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent, SequentialAgent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search, ToolContext
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part

from dotenv import load_dotenv

print(" All libraries are imported!")

 All libraries are imported!


In [11]:
load_dotenv()

True

In [12]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [13]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [14]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [ ]:
#-----------Agent Definitions for our Specialist Team MANUALLY--------------------
#agent1 
day_trip_agent = Agent(
    name = "day_trip_agent",
    model = "gemini-3.5-flash",
    description = "Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
    instruction = """
    You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

    Your Mission:
    Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

    Guidelines:
    1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
    2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
    3. **Real-Time Focus**: Search for current operating hours and special events.
    4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

    RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
    """,
    tools = [google_search]
)

#agent2
foodie_agent = Agent(
    name = "foodie_agent",
    model = "gemini-3.5-flash",
    instruction = "You are an expert food critic. Your goal is to find the absolute best food, restaurants, or culinary experiences based on a user's request. When you recommend a place, state its name clearly. For example: 'The best sushi is at **Jin Sho**.'",
)

#agent3 
weekend_guide_agent = Agent(
    name = "weekend_guide_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = "You are a local events guide. Your task is to find interesting events, concerts, festivals, and activities happening on a specific weekend."
)

#agent4
transportation_agent = Agent(
    name = "transportation_agent",
    model = "gemini-3.5-flash",
    tools = [google_search],
    instruction = "You are a navigation assistant. Given a starting point and a destination, provide clear directions on how to get from the start to the end."
)

In [ ]:
# --- The Router Agent : The Brain of the Operation ---
# We update the router's instructions to know about the new 'combo' task.
router_agent = Agent(
    name = "router_agent",
    model = "gemini-3.5-flash",
    instruction = """
     You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'foodie_agent': For queries *only* about food, restaurants, or eating.
    - 'weekend_guide_agent': For queries about events, concerts, or activities happening on a specific timeframe like a weekend.
    - 'day_trip_agent': A general planner for any other day trip requests.
    - 'find_and_navigate_combo': Use this for complex queries that ask to *first find a place* and *then get directions* to it.

    Only return the single, most appropriate option's name and nothing else.
    """
)


In [ ]:
#dict of all individual worker agents
worker_agents = {
    "day_trip_agent": day_trip_agent,
    "foodie_agent": foodie_agent,
    "weekend_guide_agent": weekend_guide_agent,
    "transportation_agent": transportation_agent, 
}
print("Agent team is assembled for the sequential workflows!")

In [ ]:
#------------------TESTING SEQUENTIAL WORKFLOW manually link agents together with custom Python code.----------------------
import re
async def run_sequential_app():
    queries = [
        "I want to eat the best sushi in Palo Alto.", # Should go to foodie_agent
        "Are there any cool outdoor concerts this weekend?", # Should go to weekend_guide_agent
        "Find me the best sushi in Palo Alto and then tell me how to get there from the Caltrain station." # Should trigger the COMBO
    ]

    for query in queries:
        print(f"\n{'='*60}\n Processing New Query: '{query}'\n{'='*60}")

        # 1. Ask the Router Agent to choose the right agent or workflow
        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        print("Asking the router agent to make a decision...")
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
        chosen_route = chosen_route.strip().replace("'", "")
        print(f" Router has selected route: '{chosen_route}'")

        # 2. Execute the chosen route
        if chosen_route == 'find_and_navigate_combo':
            print("\n--- Starting Find and Navigate Combo Workflow ---")

            # STEP 2a: Run the foodie_agent first
            foodie_session = await session_service.create_session(app_name=foodie_agent.name, user_id=my_user_id)
            foodie_response = await run_agent_query(foodie_agent, query, foodie_session, my_user_id)

            # STEP 2b: Extract the destination from the first agent's response
            # (This is a simple regex, a more robust solution might use a structured output format)
            match = re.search(r'\*\*(.*?)\*\*', foodie_response)
            if not match:
                print(" Could not determine the restaurant name from the response.")
                continue
            destination = match.group(1)
            print(f" Extracted Destination: {destination}")

            # STEP 2c: Create a new query and run the transportation_agent
            directions_query = f"Give me directions to {destination} from the Palo Alto Caltrain station."
            print(f"\n New Query for Transport Agent: '{directions_query}'")
            transport_session = await session_service.create_session(app_name=transportation_agent.name, user_id=my_user_id)
            await run_agent_query(transportation_agent, directions_query, transport_session, my_user_id)

            print("--- Combo Workflow Complete ---")

        elif chosen_route in worker_agents:
            # This is a simple, single-agent route
            worker_agent = worker_agents[chosen_route]
            worker_session = await session_service.create_session(app_name=worker_agent.name, user_id=my_user_id)
            await run_agent_query(worker_agent, query, worker_session, my_user_id)
        else:
            print(f" Error: Router chose an unknown route: '{chosen_route}'")

await run_sequential_app()

In [ ]:
'''
---------------------------EXECUTION using ADK's SEQUENTIAL WORKFLOW AGENT -------------------------------
The SequentialAgent is a workflow agent.
It's not powered by an LLM itself; instead, its only job is to execute a list of other agents in a strict, predefined order

The ADK uses a shared state dictionary that each agent in the sequence can read from and write to.

Our New Workflow:

Foodie Agent: Finds the restaurant and saves the name to state['destination'].
Transportation Agent: Automatically reads state['destination'] and uses it to find directions.
'''


